# Naive RAG — Os Sertões

Este notebook executa o mesmo código de `../naive_rag.py`: PDF → trechos → embeddings → busca por similaridade → contexto → resposta com fontes.

A busca roda localmente. A geração usa a OpenAI e precisa da chave no `.env`. Execute as células em ordem. Para evitar chamadas pagas ao abrir o notebook, `GERAR_RESPOSTAS` começa como `False`; mude para `True` quando quiser executar o lote completo.

## 1. Ambiente e configuração

Selecione o Python do ambiente `.venv` deste repositório. As instruções de instalação estão no README. Não insira sua chave nas células: configure `OPENAI_API_KEY` no `.env` local.

In [1]:
from pathlib import Path
import sys

raiz = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "naive_rag.py").is_file() and (p / "config.json").is_file()),
    None,
)
if raiz is None:
    raise FileNotFoundError("Abra o notebook dentro da pasta Naive-RAG.")
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from naive_rag import (
    Config, NaiveRAG, baixar_pdf, extrair_paginas, dividir_trechos,
    carregar_perguntas, montar_contexto, ler_json, sha256,
)

GERAR_RESPOSTAS = False
config = Config.carregar(raiz)
rag = NaiveRAG(raiz, config)
print("Projeto:", raiz)
print("Embeddings:", config.embedding_model)
print("Revisão do modelo:", config.embedding_revision)
print("Geração OpenAI habilitada:", GERAR_RESPOSTAS)

Projeto: C:\Users\fruan\projetos\Naive-RAG
Embeddings: intfloat/multilingual-e5-small
Revisão do modelo: 614241f622f53c4eeff9890bdc4f31cfecc418b3
Geração OpenAI habilitada: False


## 2. PDF e texto da obra

A edição tem 658 páginas físicas. O corpus usa as páginas PDF **33 a 653**, da nota preliminar às notas do próprio Euclides à segunda edição. Exclui a apresentação, o prefácio, o sumário e os créditos editoriais.

Nesta edição, página impressa = página física do PDF − 30. Essa relação está no `config.json` e vinculada ao hash da edição.

In [2]:
pdf = baixar_pdf(config, raiz)
paginas = extrair_paginas(pdf, config)
print("Hash do PDF:", sha256(pdf))
print("Páginas do corpus com texto:", len(paginas))
for posicao in (0, len(paginas) // 2, len(paginas) - 1):
    pagina = paginas[posicao]
    print(f"\nPDF p. {pagina['pdf_page']} / livro p. {pagina['printed_page']}")
    print(pagina["text"][:500])

Hash do PDF: c3f1dd4922d66c9bf1e7e6163882fe4b19bc45bf7a61c240aefb3b7b5dd99943
Páginas do corpus com texto: 620

PDF p. 33 / livro p. 3
nota preliminar E scrito nos raros intervalos de folga de uma carreira fatigante, este livro, que a princípio se resumia à história da Campanha de Canudos, perdeu toda a atualidade, remorada a sua publicação em virtude de causas que temos por escusado apontar. Demos-lhe, por isto, outra feição, tornando apenas variante de assunto geral o tema, a princípio dominante, que o sugeriu. Intentamos esboçar, palidamente embora, ante o olhar de futuros historiadores, os traços atuais mais expressivos das

PDF p. 343 / livro p. 313
Na plenitude do estio de novembro a março, a desolação é completa. Quem por ali se aventura tem a impressão de varar por uma roçada enorme de galhos secos e entrançados, onde a faúlha de um isqueiro ateia súbitos incêndios, se acaso estes não se alastram, espontaneamente no fastígio das secas, nos meios-dias quentes, quando o Nordeste 

## 3. Divisão em trechos

As janelas têm até 350 tokens do tokenizer do E5, com 50 tokens de sobreposição. Cada trecho pertence a uma única página, preservando fonte, posição e identificador. Os textos recebem `passage: `; as perguntas recebem `query: `, conforme as instruções do modelo.

In [3]:
amostra = dividir_trechos(paginas[:2], rag.modelo.tokenizer, config)
for trecho in amostra[:3]:
    print(trecho["id"], "tokens", trecho["token_start"], "a", trecho["token_end"])
    print(trecho["text"][:250], "\n")

C:\Users\fruan\projetos\Naive-RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


p0033-t0000 tokens 0 a 314
nota preliminar E scrito nos raros intervalos de folga de uma carreira fatigante, este livro, que a princípio se resumia à história da Campanha de Canudos, perdeu toda a atualidade, remorada a sua publicação em virtude de causas que temos por escusad 

p0034-t0000 tokens 0 a 350
lhes não permite mais a velocidade adquirida pela marcha dos povos neste século. Retardatários hoje, amanhã se extinguirão de todo. A civilização avançará nos sertões impelida por essa implacável “força motriz da História” que Gumplowicz, maior do qu 

p0034-t0300 tokens 300 a 385
o que encara a história como ela o merece: ... il s’irrite contre les demi-vérités qui sont des demi-faussetés, contre les auteurs qui n’altèrent ni une date, ni une généalogie, mais dénaturent les sentiments et les mœurs, qui gardent le dessin 1 Con 



## 4. Embeddings e índice

Os embeddings são normalizados. A busca exata usa o produto escalar, equivalente ao cosseno para esses vetores. O índice é salvo em `storage/` e inclui configuração, hash e metadados. Uma nova execução reaproveita o índice se a configuração for compatível.

In [4]:
manifesto = rag.indexar()
print("Trechos:", manifesto["total_trechos"])
print("Dimensões:", manifesto["dimensoes"])
print("Tempo da construção original:", manifesto["tempo_indexacao_s"], "s")

recarregado = NaiveRAG(raiz, config)
recarregado.carregar_indice()
assert len(recarregado.trechos) == manifesto["total_trechos"]
print("Índice recarregado com sucesso.")

Trechos: 1176
Dimensões: 384
Tempo da construção original: 229.7588 s
Índice recarregado com sucesso.


## 5. Busca e montagem do contexto

A consulta recupera até oito trechos, em ordem de similaridade. O contexto usa somente os trechos que cabem integralmente no limite de 16.000 caracteres. Não há expansão para pais nem reranking.

Similaridade não é probabilidade de a resposta estar correta. Por padrão, não aplicamos um limiar arbitrário de confiança.

In [5]:
perguntas = carregar_perguntas(raiz)
pergunta = perguntas[0]["pergunta"]
resultados = rag.buscar(pergunta)
contexto, fontes = montar_contexto(resultados, config.max_context_chars)
print(pergunta)
print("Trechos recuperados:", len(resultados))
print("Trechos no contexto:", len(fontes))
print("Caracteres do contexto:", len(contexto))
for item in fontes[:3]:
    print(f"\nPDF p. {item['pdf_page']} | livro p. {item['printed_page']} | cosseno {item['similarity']:.4f}")
    print(item["text"][:500])

Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?
Trechos recuperados: 8
Trechos no contexto: 8
Caracteres do contexto: 7704

PDF p. 123 | livro p. 93 | cosseno 0.8598
, distava afinal poucas léguas da costa. Nesta última a uberdade da terra fixara simultaneamente dois elementos, libertando o indígena. A cultura extensiva da cana, importada da Madeira, determinara o olvido dos sertões. Já antes da invasão holandesa20 do Rio Grande do Norte à Bahia havia 20 Diogo de Campos Moreno, Livro que dá razão do Estado do Brasil, séc. XVII.

PDF p. 85 | livro p. 55 | cosseno 0.8556
E o sertão é um vale fértil. É um pomar vastíssimo, sem dono. Depois tudo isto se acaba. Voltam os dias torturantes; a atmosfera asfixiadora; o empedramento do solo; a nudez da flora; e nas ocasiões em que os estios se ligam sem a intermitência das chuvas – o espasmo assombrador da seca. A natureza compraz-se em um jogo de antíteses. Eles impõ

## 6. Geração de uma resposta

O prompt orienta o modelo a usar apenas o contexto, citar `[PDF p. N]`, distinguir as interpretações históricas do autor e reconhecer informação insuficiente. A conferência automática identifica páginas citadas fora do contexto; verificar se o trecho sustenta a afirmação exige revisão humana.

In [6]:
if GERAR_RESPOSTAS:
    resposta = rag.responder(pergunta)
    print(resposta["resposta"])
    print("Conferência de citações:", resposta["citacoes"])
    print("Uso de tokens:", resposta.get("uso_tokens"))
else:
    print("Geração não executada. Configure o .env e altere GERAR_RESPOSTAS para True.")

Geração não executada. Configure o .env e altere GERAR_RESPOSTAS para True.


## 7. Executar as cinco perguntas

O lote salva JSON e Markdown em `results/`, com respostas quando habilitadas, trechos, páginas, tempos e configurações. O modo de busca produz apenas evidências; ele não representa respostas geradas nem uma avaliação final da qualidade do RAG.

In [7]:
resultado_path = rag.avaliar(somente_busca=not GERAR_RESPOSTAS, nome="notebook-respostas" if GERAR_RESPOSTAS else "notebook-busca")
execucao = ler_json(resultado_path)
print("Arquivo:", resultado_path)
for item in execucao["resultados"]:
    paginas_usadas = [fonte["pdf_page"] for fonte in item["fontes_contexto"]]
    print(item["id"], item["modo"], "páginas:", paginas_usadas)
    if item["resposta"]:
        print(item["resposta"], "\n")

Executando Q1...


Executando Q2...

Executando Q3...

Executando Q4...


Executando Q5...


Arquivo:

C:\Users\fruan\projetos\Naive-RAG\results\notebook-busca.json
Q1 somente_busca páginas: [123, 85, 95, 65, 210, 281, 277, 61]
Q2 somente_busca páginas: [143, 222, 651, 210, 94, 184, 139, 90]
Q3 somente_busca páginas: [230, 237, 34, 555, 255, 530, 138, 33]
Q4 somente_busca páginas: [230, 258, 34, 210, 207, 386, 180, 555]
Q5 somente_busca páginas: [143, 278, 123, 171, 172, 131, 132, 185]


## 8. Revisão e próximos passos

Para cada resposta, conferir cobertura da pergunta e sustentação das afirmações nos trechos. Validar as páginas no PDF e registrar insuficiências. Uma busca que retorna trechos próximos ao tema não prova que reuniu todas as evidências necessárias.

Execute `python -m unittest discover -s tests -v` para os testes locais. Para reproduzir o notebook em Python: `python naive_rag.py indexar` e `python naive_rag.py avaliar` (ou `avaliar --somente-busca`). O script e o notebook compartilham configuração e funções.

As pendências reais de desenvolvimento e revisão ficam no `resume.md` e no checklist local.